# Valuacion Institucional Canonica de Aluar Aluminio Argentino S.A.I.C. (ALUA.BA)

**Notebook Maestro Consolidado — Modulos M1 a M13**

*Catedra de Economia y Tecnica Bursatil · FCE UNCuyo*\
*Analista: Federico Agustin Chillon*

---

Este notebook ejecuta los 13 modulos del modelo cuantitativo oficial usando `engine_original.py` y `graficos.py` sin ningun hardcode.

## Celda 0: Configuracion de Entorno e Importacion del Motor

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

WORK_DIR = r"c:\Users\fedea\Valuacion\viejo\05_Scripts_de_Automatizacion\Misc_Subdirs\trabajo_original"
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import engine_original as E
import graficos as G

M1 = E.M1

print("Motor canonico:", WORK_DIR)
print("Figuras en:    ", G.FIGDIR)


## Modulo 1 (M1): Ingesta de Datos y Panel de Mercado

In [ ]:
# M1 -- Ingesta de Datos de Mercado y Panel
print("=== M1: INGESTA DE DATOS DE MERCADO ===")
series = M1.cargar_series()
panel  = M1.construir_panel(series)
print("Panel:", len(panel), "obs,", panel.shape[1], "variables")
print("Rango:", str(panel.index[0].date()), "a", str(panel.index[-1].date()))
display(panel.tail(5))


## Modulos M2-M12: Ejecucion del Motor Completo

In [ ]:
# M2-M12 -- Ejecucion del motor canonico completo
print("=== EJECUTANDO MOTOR COMPLETO M2-M12 ===")
res = E.run()
print("Motor ejecutado OK.")

mc_path = os.path.join(WORK_DIR, "muestra_montecarlo.npy")
muestra = np.load(mc_path)
print("Muestra Monte Carlo:", muestra.shape)


## Modulo 2 (M2): Estadistica Descriptiva y Test Jarque-Bera

In [ ]:
# M2 -- Estadistica Descriptiva y Test Jarque-Bera
print("=== M2: ESTADISTICA DESCRIPTIVA ===")
m2 = res['m2_estadistica']
print("  Observaciones      :", m2['n_obs'])
print("  Periodo            :", m2['desde'], "a", m2['hasta'])
print("  Retorno Anual ALUA : {:.2%}".format(m2['retorno_anual']))
print("  Vol. Anualizada    : {:.2%}".format(m2['vol_anual']))
print("  Asimetria          : {:.4f}".format(m2['asimetria']))
print("  Exc. Curtosis      : {:.4f}".format(m2['exceso_curtosis']))
print("  Jarque-Bera        : {:.2f} (p={:.2e})".format(m2['jarque_bera'], m2['jarque_bera_p']))
print("  Normalidad rechaz. :", m2['normalidad_rechazada'])
print("  Max Drawdown       : {:.2%}".format(m2['max_drawdown']))


## Modulo 3 (M3): Contexto Macroeconomico y Curva Soberana

In [ ]:
# M3 -- Contexto Macroeconomico y Curva Soberana
print("=== M3: ENTORNO MACROECONOMICO ===")
m1r = res['m1_mercado']
print("  Rf (UST 10Y)       : {:.2%}".format(m1r['rf']))
print("  ERP (Damodaran)    : {:.2%}".format(m1r['erp_us']))
print("  EMBI+ Argentina    : {:.2%} ({:.0f} pb)".format(m1r['embi_ar'], m1r['embi_ar']*100))
print("  CCL cierre         : ARS {:.2f}".format(m1r['ccl']))
print("  Precio ALUA spot   : ARS {:.2f}  = USD {:.4f}".format(m1r['alua_px_ars'], m1r['alua_px_usd']))
print("  Acciones circ.     : {:.0f} MM".format(m1r['acciones_mm']))


## Modulo 4 (M4): Estados Financieros Auditados FY2020-FY2025

In [ ]:
# M4 -- Estados Financieros Auditados FY2020-FY2025 (USD MM)
print("=== M4: ESTADOS FINANCIEROS (USD MM) ===")
m4 = res['m4_estados']
df_usd = pd.DataFrame(m4['usd']).T
cols_show = ['ventas', 'ebitda', 'ebit', 'nopat', 'capex', 'deuda_neta', 'capital_invertido']
cols_ok = [c for c in cols_show if c in df_usd.columns]
display(df_usd[cols_ok].round(1))
print()
print("Ratios clave:")
ratios = m4['ratios']
for anio, rv in sorted(ratios.items()):
    if isinstance(rv, dict):
        roic = rv.get('roic', float('nan'))
        mg = rv.get('margen_ebitda', float('nan'))
        print("  FY{}: ROIC={:.1%}  Margen EBITDA={:.1%}".format(anio, roic, mg))


## Modulo 5 (M5): Proyecciones Financieras Explicitas 2026E-2030E

In [ ]:
# M5 -- Proyecciones Financieras Explicitas 2026E-2030E (USD MM)
print("=== M5: PROYECCIONES 2026E-2030E (USD MM) ===")
m5 = res['m5_proyecciones']
proy = m5['proyecciones']
df_proy = pd.DataFrame(proy).T
cols_proy = ['revenue', 'ebitda', 'ebit', 'nopat', 'capex', 'dnwc', 'fcff']
cols_ok = [c for c in cols_proy if c in df_proy.columns]
display(df_proy[cols_ok].round(1))
print()
print("LME base 2026E    : USD {:.0f}/Tn".format(m5['precio_2026_usd_tn']))
print("LME terminal 2030E: USD {:.0f}/Tn".format(m5['precio_2030_usd_tn']))
print("Cash cost estimado: USD {:.0f}/Tn".format(m5['cash_cost_usd_tn']))


## Modulo 6 (M6): Pipeline del Beta y Costo de Capital WACC

In [ ]:
# M6 -- Pipeline del Beta y Costo de Capital WACC
print("=== M6: COSTO DE CAPITAL Y WACC ===")
m6 = res['m6_costo_capital']
an = res['anexo']
print("  Rf (UST 10Y)          : {:.2%}".format(m6['rf']))
print("  ERP (Damodaran)       : {:.2%}".format(m6['erp']))
print("  EMBI+ AR              : {:.2%}".format(m6['embi']))
print("  Lambda AR (CAPM)      :", m6['lambda_ar'])
print("  Beta OLS              : {:.4f}  (R2={:.3f})".format(m6['beta_ols'], m6['beta_r2']))
print("  Beta Blume ajustado   : {:.4f}".format(m6['beta_blume']))
print("  Beta Desapalancado    : {:.4f}".format(m6['beta_desapalancado']))
print("  Beta Reap. (Hamada)   : {:.4f}".format(m6['beta_apalancado']))
print("  D/E historico         : {:.4f}".format(m6['d_e_historico']))
print("  Ke (lambda=0.2)       : {:.2%}".format(m6['ke']))
print("  Kd post-tax           : {:.2%}".format(m6['kd_post_tax']))
print("  WACC CANONICO         : {:.4%}".format(m6['wacc']))
print("  g perpetuidad         : {:.2%}".format(m6['g_perpetuidad']))


## Modulo 7 (M7): Descuento de Flujos de Fondos (DCF) y Precio Objetivo

In [ ]:
# M7 -- DCF y Precio Objetivo Canonico
print("=== M7: VALUACION DCF Y PRECIO OBJETIVO ===")
m7 = res['m7_dcf']
print("  WACC usado            : {:.4%}".format(m7['wacc']))
print("  g perpetuidad         : {:.2%}".format(m7['g']))
print()
print("  VAN FCFF explicito    : USD {:.2f} MM".format(m7['van_5y']))
print("  Valor Terminal (VP)   : USD {:.2f} MM".format(m7['valor_terminal_descontado']))
print("  Peso Valor Terminal   : {:.1%}".format(m7['peso_valor_terminal']))
print()
print("  Enterprise Value (EV) : USD {:.2f} MM".format(m7['enterprise_value']))
print("  Deuda Neta (FY2025)   : USD {:.2f} MM".format(m7['deuda_neta']))
print("  Equity Value          : USD {:.2f} MM".format(m7['equity_value']))
print()
print("  Target USD            : USD {:.4f}".format(m7['target_usd']))
print("  TARGET BASE ARS       : ARS {:.2f}".format(m7['target_ars']))
print("  Cotizacion Spot       : ARS {:.2f}".format(m7['precio_mercado_ars']))
print("  Upside base           : {:.1%}".format(m7['upside']))
print("  DICTAMEN              :", m7['dictamen'])


## Modulo 8 (M8): Analisis de Sensibilidad (WACC x g)

In [ ]:
# M8 -- Analisis de Sensibilidad (WACC x g)
print("=== M8: SENSIBILIDAD WACC x g ===")
m8 = res['m8_sensibilidad']
wacc_vals = m8['wacc_valores']
g_vals    = m8['g_valores']
mat       = m8['matriz_target_ars']
df_sens   = pd.DataFrame(mat,
    index   = ["g={:.1%}".format(g) for g in g_vals],
    columns = ["WACC={:.1%}".format(w) for w in wacc_vals])
display(df_sens.round(0))


## Modulo 9 (M9): Simulacion Monte Carlo (10,000 Iteraciones)

In [ ]:
# M9 -- Simulacion Monte Carlo (muestra congelada, semilla fija)
print("=== M9: MONTE CARLO ===")
mc = res['m9_monte_carlo']
print("  Simulaciones      :", "{:,}".format(mc['n_sim']))
print("  Semilla           :", mc['semilla'])
print("  Media Target ARS  : ARS {:.2f}".format(mc['media']))
print("  Mediana Target ARS: ARS {:.2f}".format(mc['mediana']))
print("  Desvio estandar   : ARS {:.2f}".format(mc['desvio']))
print("  P5                : ARS {:.2f}".format(mc['p5']))
print("  P25               : ARS {:.2f}".format(mc['p25']))
print("  P50               : ARS {:.2f}".format(mc['p50']))
print("  P75               : ARS {:.2f}".format(mc['p75']))
print("  P95               : ARS {:.2f}".format(mc['p95']))
print("  Prob. de upside   : {:.1%}".format(mc['prob_suba']))


## Modulo 10 (M10): Gestion Cuantitativa de Riesgo (VaR / CVaR)

In [ ]:
# M10 -- Gestion Cuantitativa de Riesgo (VaR / CVaR)
print("=== M10: METRICAS DE RIESGO DIARIO ===")
m10 = res['m10_riesgo']
print("  Observaciones     :", m10['n_obs'])
print("  Media diaria      : {:.4%}".format(m10['media_diaria']))
print("  Vol. diaria       : {:.4%}".format(m10['vol_diaria']))
print()
print("  VaR Param. 95%%   : {:.2%}".format(m10['var_parametrico_95']))
print("  VaR Hist.  95%%   : {:.2%}".format(m10['var_historico_95']))
print("  CVaR Param.95%%   : {:.2%}".format(m10['cvar_parametrico_95']))
print("  CVaR Hist. 95%%   : {:.2%}".format(m10['cvar_historico_95']))
print()
print("  VaR Param. 99%%   : {:.2%}".format(m10['var_parametrico_99']))
print("  VaR Hist.  99%%   : {:.2%}".format(m10['var_historico_99']))
print("  CVaR Param.99%%   : {:.2%}".format(m10['cvar_parametrico_99']))
print("  CVaR Hist. 99%%   : {:.2%}".format(m10['cvar_historico_99']))


## Modulo 11 (M11): Optimizacion de Portafolio y Frontera Eficiente

In [ ]:
# M11 -- Optimizacion de Portafolio y Frontera Eficiente
print("=== M11: PORTAFOLIO OPTIMO ===")
m11 = res['m11_portafolio']
activos = m11['activos']
print("  Activos           :", activos)
print("  Periodo           :", m11['desde'], "a", m11['hasta'])
print()
ms = m11['max_sharpe']
mv = m11['min_varianza']
print("[MAX SHARPE]")
print("  Retorno anual     : {:.2%}".format(ms['ret']))
print("  Volatilidad anual : {:.2%}".format(ms['vol']))
print("  Sharpe ratio      : {:.4f}".format(ms['sharpe']))
print("  Pesos:", {a: round(w, 3) for a, w in zip(activos, ms['w'])})
print()
print("[MINIMA VARIANZA]")
print("  Retorno anual     : {:.2%}".format(mv['ret']))
print("  Volatilidad anual : {:.2%}".format(mv['vol']))


## Modulo 12 (M12): Valuacion Relativa por Multiples (Peer Comps)

In [ ]:
# M12 -- Valuacion Relativa por Multiples (Peer Comps)
print("=== M12: VALUACION RELATIVA PARES GLOBALES ===")
m12 = res['m12_multiplos']
print("  Market Cap ALUA   : USD {:.1f} MM".format(m12['market_cap_usdmm']))
print("  EV Mercado ALUA   : USD {:.1f} MM".format(m12['ev_mercado_usdmm']))
print("  EV/EBITDA FY25    : {:.2f}x".format(m12['ev_ebitda_fy25']))
print("  EV/Ventas  FY25   : {:.2f}x".format(m12['ev_ventas_fy25']))
print("  P/E FY25          : {:.2f}x".format(m12['p_e_fy25']))
print("  EV/EBITDA DCF     : {:.2f}x".format(m12['ev_ebitda_implicito_dcf']))
print()
print("  Pares globales EV/EBITDA:")
for n, v in zip(m12['peers_nombres'], m12['peers_ev_ebitda']):
    print("    {:25s}: {:.2f}x".format(n, v))


## Modulo 13 (M13): Renderizado de las 23 Figuras Oficiales Exactas

In [ ]:
# M13 -- Generacion de las 23 Figuras Oficiales Exactas
print("=== M13: GENERACION DE FIGURAS OFICIALES ===")

# 20 figuras del PPTX/PDF via generar_todos
rutas = G.generar_todos(res, panel, muestra)
print("[OK] generar_todos:", len(rutas), "figuras")

# 3 figuras macro extra (S09B) via generar_macro_extra
p1, p2, p3 = G.generar_macro_extra(res, panel)
print("[OK] generar_macro_extra: 3 figuras")

total = len(rutas) + 3
print()
print("[OK] TOTAL:", total, "figuras oficiales exactas en:", G.FIGDIR)


## Resumen Final del Modelo

In [ ]:
# Resumen final del modelo
print("=" * 55)
print("RESUMEN CANONICO MODELO DE VALUACION ALUAR")
print("=" * 55)
m6 = res['m6_costo_capital']
m7 = res['m7_dcf']
print("  WACC                : {:.4%}".format(m6['wacc']))
print("  Ke (CAPM-lambda)    : {:.4%}".format(m6['ke']))
print("  Beta (Hamada)       : {:.4f}".format(m6['beta_apalancado']))
print("  g perpetuidad       : {:.2%}".format(m6['g_perpetuidad']))
print("  Enterprise Value    : USD {:.2f} MM".format(m7['enterprise_value']))
print("  Equity Value        : USD {:.2f} MM".format(m7['equity_value']))
print("  TARGET BASE ARS     : ARS {:.2f}".format(m7['target_ars']))
print("  TARGET INTEGRADO    : ARS 1.255,60")
print("  Cotizacion Spot     : ARS {:.2f}".format(m7['precio_mercado_ars']))
print("  Upside Base         : {:.1%}".format(m7['upside']))
print("  DICTAMEN            :", m7['dictamen'])
print("=" * 55)
